# Grocery Product Demand Analysis
**Course:** Exploratory Data Analysis and Visualization (EDAV)  
**Dataset:** Supermarket Sales CSV  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

## Objective
Explore transaction-level supermarket sales to understand product-line demand, branch performance, customer spending, time patterns, and payment preferences. Fill in your name, roll number, department, and submission date.

**Dataset limitation:** This dataset supports descriptive analysis of transactions and quantity sold. It does not include inventory or stockout data, so it cannot measure unmet demand or provide a reliable inventory forecast.

## 1. Load the single CSV file
Set `DATA_PATH` to the exact path of your downloaded CSV. This notebook does not use a ZIP file or a manual upload dialog.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

# If the file is in Colab's sample_data folder, use this path:
# If it is in Google Drive, mount Drive and replace the path above:
from google.colab import drive
drive.mount("/content/drive")
DATA_PATH = "/content/drive/MyDrive/supermarket_sales.csv"
if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(f"CSV not found: {DATA_PATH}. Update DATA_PATH to the exact file location.")
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df), " Columns:", len(df.columns))
display(df.head())

## 2. Inspect and understand the dataset

In [ ]:
print("Columns:", df.columns.tolist())
df.info()
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())

## 3. Data cleaning and preparation

In [ ]:
df = df.drop_duplicates().copy()
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
if "Time" in df.columns:
    parsed_time = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce")
    df["Hour"] = parsed_time.dt.hour
for col in ["Unit price", "Quantity", "Tax 5%", "Total", "cogs", "gross margin percentage", "gross income", "Rating"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print("Shape after duplicate removal:", df.shape)
display(df.isna().sum().to_frame("Missing values"))

## 4. General EDA: distributions of sales, quantity, rating, and income

In [ ]:
for col in ["Total", "Quantity", "Rating", "gross income"]:
    if col in df.columns:
        plt.figure(figsize=(7, 4))
        sns.histplot(df[col].dropna(), kde=True)
        plt.title(f"Distribution of {col}")
        plt.tight_layout()
        plt.show()

## 5. Question-wise analysis
The following questions are mapped to CO/BL levels for the course project. Confirm the exact mapping required by your instructor.

### Question 1 [CO1, BL3]: Which product lines have the highest quantity sold and total sales?

In [ ]:
if {"Product line", "Quantity", "Total"}.issubset(df.columns):
    product_summary = df.groupby("Product line").agg(
        Total_Quantity=("Quantity", "sum"), Total_Sales=("Total", "sum"),
        Average_Transaction=("Total", "mean"), Transactions=("Total", "size")
    ).sort_values("Total_Quantity", ascending=False)
    display(product_summary.round(2))
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    product_summary["Total_Quantity"].sort_values().plot.barh(ax=ax[0], title="Quantity Sold by Product Line")
    product_summary["Total_Sales"].sort_values().plot.barh(ax=ax[1], title="Total Sales by Product Line")
    plt.tight_layout(); plt.show()
else:
    print("Required columns missing. Check df.columns.tolist().")

### Question 2 [CO2, BL3]: How do sales and quantity vary across branches?

In [ ]:
if {"Branch", "Total", "Quantity"}.issubset(df.columns):
    branch_summary = df.groupby("Branch").agg(Total_Sales=("Total", "sum"), Total_Quantity=("Quantity", "sum"), Transactions=("Total", "size")).sort_values("Total_Sales", ascending=False)
    display(branch_summary.round(2))
    sns.barplot(data=df, x="Branch", y="Total", estimator=sum, errorbar=None)
    plt.title("Total Sales by Branch"); plt.tight_layout(); plt.show()
else:
    print("Branch/Total/Quantity column missing.")

### Question 3 [CO3, BL4]: Do average transaction totals differ by customer type and gender?

In [ ]:
if {"Customer type", "Gender", "Total"}.issubset(df.columns):
    summary = df.groupby(["Customer type", "Gender"]).agg(Transactions=("Total", "size"), Average_Spend=("Total", "mean"), Total_Sales=("Total", "sum")).round(2)
    display(summary)
    sns.barplot(data=df, x="Customer type", y="Total", hue="Gender", errorbar=None)
    plt.title("Average Transaction Total by Customer Type and Gender"); plt.tight_layout(); plt.show()
else:
    print("Customer type/Gender/Total column missing.")

### Question 4 [CO4, BL4]: How do sales and transaction counts vary by date and hour?

In [ ]:
if {"Date", "Total"}.issubset(df.columns):
    daily = df.dropna(subset=["Date"]).groupby(df["Date"].dt.date).agg(Daily_Sales=("Total", "sum"), Transactions=("Total", "size"))
    display(daily.head())
    daily["Daily_Sales"].plot(figsize=(12, 4), marker="o")
    plt.title("Daily Total Sales"); plt.ylabel("Sales"); plt.tight_layout(); plt.show()
if {"Hour", "Total"}.issubset(df.columns) and df["Hour"].notna().any():
    hourly = df.groupby("Hour").agg(Transactions=("Total", "size"), Sales=("Total", "sum"))
    display(hourly)
    hourly.plot(kind="bar", subplots=True, figsize=(11, 7), legend=False, title=["Transactions by Hour", "Sales by Hour"])
    plt.tight_layout(); plt.show()

### Question 5 [CO5, BL5]: Which payment methods are most used, and how do sales compare?

In [ ]:
if {"Payment", "Total"}.issubset(df.columns):
    payment_summary = df.groupby("Payment").agg(Transactions=("Total", "size"), Total_Sales=("Total", "sum"), Average_Transaction=("Total", "mean")).sort_values("Transactions", ascending=False).round(2)
    display(payment_summary)
    payment_summary[["Transactions", "Total_Sales"]].plot(kind="bar", subplots=True, figsize=(10, 7), legend=False, title=["Transactions by Payment Method", "Total Sales by Payment Method"])
    plt.tight_layout(); plt.show()
else:
    print("Payment or Total column missing.")

## 6. Correlation analysis

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(10, 7))
    sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap of Numeric Variables")
    plt.tight_layout(); plt.show()
else:
    print("Not enough numeric columns for correlation analysis.")

## 7. Automatically generated key findings

In [ ]:
if {"Product line", "Quantity", "Total"}.issubset(df.columns):
    q = df.groupby("Product line")["Quantity"].sum().sort_values(ascending=False)
    s = df.groupby("Product line")["Total"].sum().sort_values(ascending=False)
    print(f"Highest quantity product line: {q.index[0]} ({q.iloc[0]:,.0f} units)")
    print(f"Highest sales product line: {s.index[0]} ({s.iloc[0]:,.2f})")
if {"Branch", "Total"}.issubset(df.columns):
    b = df.groupby("Branch")["Total"].sum().sort_values(ascending=False)
    print(f"Highest sales branch: {b.index[0]} ({b.iloc[0]:,.2f})")
if {"Payment", "Total"}.issubset(df.columns):
    p = df["Payment"].value_counts()
    print(f"Most common payment method: {p.index[0]} ({p.iloc[0]} transactions)")

## 8. Results, conclusion, and recommendations
After running all cells, write your interpretation using the actual tables and charts. Include: (1) strongest product lines by quantity and sales, (2) differences across branches, (3) observed customer spending differences, (4) peak observed date/hour patterns, and (5) most common payment method. Recommendations should be supported by the results.

**Limitations:** The dataset has a limited transaction history and lacks inventory, stockout, and customer-level longitudinal data. Findings describe observed sales; they do not prove causes or forecast future demand.

## 9. References
1. Dataset page: https://www.kaggle.com/datasets/aungpyaeap/supermarket-sales
2. Pandas documentation: https://pandas.pydata.org/docs/
3. Matplotlib documentation: https://matplotlib.org/stable/
4. Seaborn documentation: https://seaborn.pydata.org/

## Troubleshooting
- `FileNotFoundError`: update `DATA_PATH` to the actual location and exact filename.
- If the CSV columns differ, run `print(df.columns.tolist())` and update the relevant column names.
- Run cells from top to bottom after setting the correct path.